In [1]:
import pandas as pd
import os
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

C:\Users\Dell\Desktop\ML\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ROOT_DIR = os.path.abspath(os.path.join(os.path.join(os.getcwd(), ".."), ".."))
DATA_DIR = os.path.join(ROOT_DIR, "Data")
DATASET_DIR = os.path.join(DATA_DIR, "SMS-Spam-Data")
DATASET_FILE = os.path.join(DATASET_DIR, "Dataset_5971.csv")

In [3]:
df = pd.read_csv(DATASET_FILE)
df.head()

,LABEL,TEXT,URL,EMAIL,PHONE
0,ham,Your opinion about me? 1. Over 2. Jada 3. Kusr...,No,No,No
1,ham,What's up? Do you want me to come online? If y...,No,No,No
2,ham,So u workin overtime nigpun?,No,No,No
3,ham,"Also sir, i sent you an email about how to log...",No,No,No
4,Smishing,Please Stay At Home. To encourage the notion o...,No,No,No


In [4]:
df["TEXT"] = df["TEXT"].str.lstrip("\t")
df["CLEAN"] = df["TEXT"].str.lower().str.split().str.join(" ")
df["LABEL"] = df["LABEL"].str.strip().str.lower()
df = df.drop_duplicates(subset="CLEAN").drop(columns="CLEAN")
df.shape

(5831, 5)

## Raw text, no `<PHONE>`/`<URL>` masking

Unlike the Baseline/NN/CNN notebooks, this one skips the placeholder masking. A pretrained
sentence encoder has already seen phone numbers and URLs in its own pretraining data and
tokenizes them fine on its own; masking them to `<PHONE>`/`<URL>` would throw away
information the encoder could otherwise use, for no benefit specific to this model.

In [5]:
X = df["TEXT"]
y = df["LABEL"]

In [6]:
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1, stratify=y_temp, random_state=42)

In [7]:
encoder = LabelEncoder()
y_train_encoded = encoder.fit_transform(y_train)
y_val_encoded = encoder.transform(y_val)
y_test_encoded = encoder.transform(y_test)

## Frozen sentence embeddings

`all-MiniLM-L6-v2` as a feature extractor only - no fine-tuning, no gradient ever touches
its weights. First run downloads the model from the Hugging Face Hub (~90MB, cached
locally afterward, needs internet once).

In [8]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

X_train_emb = embedder.encode(X_train.tolist(), show_progress_bar=True)
X_val_emb = embedder.encode(X_val.tolist(), show_progress_bar=True)
X_test_emb = embedder.encode(X_test.tolist(), show_progress_bar=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4891.12it/s]

Batches:   0%|          | 0/132 [00:00<?, ?it/s]

Batches:   1%|          | 1/132 [00:00<02:06,  1.04it/s]

Batches:   2%|▏         | 2/132 [00:01<01:17,  1.68it/s]

...

Batches:  89%|████████▉ | 33/37 [00:13<00:00,  5.86it/s]

Batches:  92%|█████████▏| 34/37 [00:13<00:00,  6.02it/s]

Batches:  95%|█████████▍| 35/37 [00:13<00:00,  6.12it/s]

Batches:  97%|█████████▋| 36/37 [00:13<00:00,  6.44it/s]

Batches: 100%|██████████| 37/37 [00:13<00:00,  2.74it/s]

In [9]:
model = LogisticRegression(class_weight="balanced", max_iter=1000)
model.fit(X_train_emb, y_train_encoded)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Defaul

In [10]:
y_val_pred = model.predict(X_val_emb)
print(classification_report(y_val_encoded, y_val_pred, target_names=encoder.classes_, zero_division=0, digits=3))

              precision    recall  f1-score   support

         ham      0.982     0.972     0.977       387
    smishing      0.881     0.841     0.860        44
        spam      0.643     0.750     0.692        36

    accuracy                          0.942       467
   macro avg      0.835     0.854     0.843       467
weighted avg      0.946     0.942     0.944       467



In [11]:
y_pred = model.predict(X_test_emb)
print(classification_report(y_test_encoded, y_pred, target_names=encoder.classes_, zero_division=0, digits=3))

              precision    recall  f1-score   support

         ham      0.992     0.967     0.979       967
    smishing      0.783     0.927     0.849       109
        spam      0.737     0.769     0.753        91

    accuracy                          0.948      1167
   macro avg      0.837     0.888     0.860      1167
weighted avg      0.952     0.948     0.949      1167



In [12]:
conf_matrix = confusion_matrix(y_test_encoded, y_pred)
print(conf_matrix)

[[935  14  18]
 [  1 101   7]
 [  7  14  70]]


In [13]:
results = pd.DataFrame({
    "text": X_test.to_numpy(),
    "true": encoder.inverse_transform(y_test_encoded),
    "pred": encoder.inverse_transform(y_pred),
})

error = results[results["true"] != results["pred"]]
error

,text,true,pred
12,Plz note: if anyone calling from a mobile Co. ...,ham,smishing
25,Can you plz tell me the answer. Reeta sent via...,ham,spam
82,Had the money issue weigh me down but thanks t...,ham,spam
98,ROMCA Everyone around should be responding wel...,spam,ham
125,No message..no response..what happend? Any iss...,ham,smishing
...,...,...,...
1088,No problem with the renewal. I.ll do it right ...,ham,spam
1108,Do you want a New Nokia 3510i colour phone Del...,spam,smishing
1109,Thank You for calling.Forgot to say Happy Onam...,ham,smishing
1128,We know someone who you know that fancies you....,spam,smishing
